In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
for i in df.columns:
  if df[i].isnull().sum() > 100:
    df.drop(columns=[i],inplace=True)

for i in df.columns:
  df[i].fillna(df[i].median(), inplace=True)

In [ ]:
df.describe()

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)
  print("Duplicates Dropped.")
else:
  print("No Duplicate Samples Found.")

In [ ]:
# Task 3: Write your code here:
print(df.select_dtypes(include=["object"]).columns)
print("No categorical columns found")

In [ ]:
# Task 4: Write your code here:
cols = df.columns.drop("Target")
scaler = StandardScaler()
df[cols] = pd.DataFrame(scaler.fit_transform(df[cols]), columns=cols)
df.head()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Target')
plt.title('Distribution of Target Variable')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
print("Clearly imbalanced data")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits=5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=67)

catBoost = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=6
  )

metrics = {'accuracy': [], 'f1': []}
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  print("Training CatBoost...")
  catBoost.fit(X_train, y_train)
  y_pred = catBoost.predict(X_test)

  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  metrics['accuracy'].append(accuracy)
  metrics['f1'].append(f1)

print(f" Accuracy: {np.mean(metrics["accuracy"]):.4f}")
print(f" F1-Score: {np.mean(metrics["f1"]):.4f}")

In [ ]:
majority_class = y.value_counts().idxmax()
baseline_pred = [majority_class] * len(y)

# Evaluate the baseline
baseline_accuracy = accuracy_score(y, baseline_pred)
baseline_f1 = f1_score(y, baseline_pred, average='weighted', zero_division=0)

print(f"Baseline Accuracy (majority class): {baseline_accuracy:.4f}")
print(f"Baseline F1-Score: {baseline_f1:.4f}")

In [ ]:
# Task 1: Write your code here:
feature_cols = df.columns.drop("Target")
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': catBoost.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 40))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print("D_45 is clearly the golden feature")

In [ ]:
# Task Bonus: Write your code here:
X = df[["D_45"]]
n_splits=5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=67)

catBoost = CatBoostClassifier(
      verbose=0,
      n_estimators=100,
      max_depth=6
  )

metricsB = {'accuracy': [], 'f1': []}
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  print("Training CatBoost...")
  catBoost.fit(X_train, y_train)
  y_pred = catBoost.predict(X_test)

  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  metricsB['accuracy'].append(accuracy)
  metricsB['f1'].append(f1)

print(f"\nGolden model:\nAccuracy: {np.mean(metricsB["accuracy"]):.4f}")
print(f"F1-Score: {np.mean(metricsB["f1"]):.4f}")
print(f"\nFull model:\nAccuracy: {np.mean(metrics["accuracy"]):.4f}")
print(f"F1-Score: {np.mean(metrics["f1"]):.4f}")